# A/B-тест геймификации

**Автор:** Сергей  
**Дата:** 2026-05-24  
**Период теста:** 2026-05-04 — 2026-05-24 (20 дней)  
**Данные:** ab_test_users_groups.csv, ab_test_events.csv

**Цель:** оценить влияние фичи (геймификация) на вовлечённость и удержание пользователей.

**Гипотеза:** фича повышает Completion Rate и Retention 7/14 в группе B относительно группы A.

## Методика

**Primary-метрики:** Retention 7, Retention 14.

**Secondary-метрика:** Completion Rate.

**Тест:** z-тест для двух пропорций (pooled variance), двусторонний, α = 0.05.

**Поправка на множественные сравнения:** не применялась. Обоснование: худшее p-value среди основных метрик (4.2e-04) на порядок ниже α/3 = 0.0167, поэтому любая стандартная поправка (Бонферрони, Холм, FDR) не меняет решения. Метрики измерены на одних пользователях и коррелированы, что делает Бонферрони избыточно консервативным.

**Подгруппы по стажу:** проверка устойчивости знака эффекта, формальные тесты внутри подгрупп не проводились.

## Загрузка данных

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

df_ev = pd.read_csv('data/ab_test_events.csv')
df_ug = pd.read_csv('data/ab_test_users_groups.csv')

## Первичная разведка

Проверяем: размеры, типы, пропуски, диапазон дат, пересечения между таблицами.

In [2]:
df_ug.shape, df_ev.shape

((6008, 4), (179330, 3))

In [3]:
df_ug.dtypes, df_ev.dtypes

(user_id          str
 group            str
 signup_date      str
 grade          int64
 dtype: object,
 user_id       str
 timestamp     str
 event_type    str
 dtype: object)

In [4]:
df_ug.isna().sum(), df_ev.isna().sum()

(user_id        0
 group          0
 signup_date    0
 grade          0
 dtype: int64,
 user_id       0
 timestamp     0
 event_type    0
 dtype: int64)

In [5]:
df_ug.value_counts('group'), df_ug['user_id'].nunique()

(group
 A    3005
 B    3003
 Name: count, dtype: int64,
 6000)

In [6]:
df_ev.value_counts('event_type')

event_type
login                 75552
homework_started      68317
homework_completed    28995
return_visit_d7        4519
return_visit_d14       1947
Name: count, dtype: int64

In [7]:
print('signup_date:', df_ug['signup_date'].min(), df_ug['signup_date'].max())
print('timestamp:', df_ev['timestamp'].min(), df_ev['timestamp'].max())

signup_date: 2025-03-31 2026-04-04
timestamp: 2026-05-04 00:06:44 2026-05-24 23:55:43


## Очистка данных

Три дефекта: дубли user_id (порча рандомизации), дубли событий (баг логирования), неполное окно d14.

### Дубли user_id

8 пользователей попали одновременно в группы A и B, сбой рандомизации. Исключаем их из обеих таблиц.

In [8]:
b_id = df_ug['user_id'].value_counts()
bad = b_id[b_id > 1]
df_ug[df_ug['user_id'].isin(bad.index)].sort_values('user_id')

,user_id,group,signup_date,grade
357,U300357,A,2026-03-19,9
6000,U300357,B,2026-03-19,9
460,U300460,B,2025-08-12,6
6001,U300460,A,2025-08-12,6
920,U300920,B,2025-04-09,10
6002,U300920,A,2025-04-09,10
1014,U301014,A,2026-03-28,10
6003,U301014,B,2026-03-28,10
1087,U301087,B,2025-06-27,10
6004,U301087,A,2025-06-27,10


In [9]:
df_ug = df_ug[~df_ug['user_id'].isin(bad.index)]
df_ug.value_counts('group')

group
A    2997
B    2995
Name: count, dtype: int64

### Дубли событий

2 полных дубля событий, баг логирования. Удаляем.

In [10]:
df_ev.duplicated().sum()

np.int64(2)

In [11]:
df_ev = df_ev[~df_ev['user_id'].isin(bad.index)]
df_ev = df_ev.drop_duplicates(['user_id', 'timestamp', 'event_type'])
df_ev.shape

(179111, 3)

### Неполное окно d14

35 пользователей с первым событием позже 2026-05-10, у них не наступило полное 14-дневное окно. Исключаем из знаменателя Retention 14, но оставляем в других метриках.

In [12]:
df_ev['timestamp'] = df_ev['timestamp'].astype('datetime64[ns]')
df_ug['signup_date'] = df_ug['signup_date'].astype('datetime64[ns]')

d14 = df_ev.groupby('user_id')['timestamp'].min() > '2026-05-10'
d14T = d14[d14 == True]
len(d14T.index)

35

In [13]:
d14z = df_ug[~df_ug['user_id'].isin(d14T.index)]
d14z['group'].value_counts()

group
A    2979
B    2978
Name: count, dtype: int64

## Объединённый датасет

События плюс характеристики пользователей (группа, signup_date, grade).

In [14]:
df_eg = df_ev.merge(df_ug, how='left')
df_eg.shape, df_eg['group'].value_counts()

((179111, 6),
 group
 B    96198
 A    82913
 Name: count, dtype: int64)

## Характеристики групп

Проверяем баланс: размеры (SRM), распределение по grade, распределение по стажу.

### SRM

Проверка, что разбиение на группы не сломано: ожидаем 50/50.

In [15]:
stats.chisquare([2997, 2995], f_exp=[2996, 2996])

Power_divergenceResult(statistic=np.float64(0.0006675567423230974), pvalue=np.float64(0.9793872546704734))

### Баланс по grade

Сопоставимость распределения оценок между группами.

In [16]:
cst = pd.crosstab(df_ug['grade'], df_ug['group'])
cst

group,A,B
grade,,
5,446,416
6,447,465
7,403,437
8,454,432
9,431,379
10,410,438
11,406,428


In [17]:
stats.chi2_contingency(cst)

Chi2ContingencyResult(statistic=np.float64(8.1642815468651), pvalue=np.float64(0.2263130493297412), dof=6, expected_freq=array([[431.14385848, 430.85614152],
       [456.15220294, 455.84779706],
       [420.14018692, 419.85981308],
       [443.14786382, 442.85213618],
       [405.13518024, 404.86481976],
       [424.14152203, 423.85847797],
       [417.13918558, 416.86081442]]))

### Баланс по стажу

Стаж считаем как `2026-05-04` минус `signup_date`. Бакеты: менее 90 дней (новички), 90–180 (продвинутые), более 180 (опытные).

In [18]:
start = pd.Timestamp('2026-05-04')
signup_date = df_ug['signup_date']
tenure_days = (start - signup_date).dt.days

buckets = pd.cut(tenure_days, bins=[0, 90, 180, np.inf], labels=['новички', 'продвинутые', 'опытные'])
df_ug['tenure_bucket'] = buckets
df_ug['tenure_bucket'].value_counts()

tenure_bucket
опытные        3553
продвинутые    1455
новички         984
Name: count, dtype: int64

In [19]:
buc_tab = pd.crosstab(buckets, df_ug['group'])
buc_tab

group,A,B
signup_date,,
новички,456,528
продвинутые,761,694
опытные,1780,1773


In [20]:
stats.chi2_contingency(buc_tab)

Chi2ContingencyResult(statistic=np.float64(8.366640588390768), pvalue=np.float64(0.015247796255677815), dof=2, expected_freq=array([[ 492.16421896,  491.83578104],
       [ 727.74282377,  727.25717623],
       [1777.09295728, 1775.90704272]]))

## Целевые метрики

Считаем три метрики по группам A и B.

### Completion Rate

In [21]:
hwc = df_eg[df_eg['event_type'] == 'homework_completed']
hwc1 = hwc.groupby('group')['user_id'].nunique()
denom = {'A': 2997, 'B': 2995}
completion_rate = hwc1 / hwc1.index.map(denom)
completion_rate

group
A    0.976310
B    0.988314
dtype: float64

### Retention 7

Знаменатель: все пользователи групп A и B.

In [22]:
d7 = df_eg[df_eg['event_type'] == 'return_visit_d7']
d7z = d7.groupby('group')['user_id'].nunique()
denom = {'A': 2997, 'B': 2995}
d7z / d7z.index.map(denom)

group
A    0.727060
B    0.779633
dtype: float64

### Retention 14

Знаменатель: 2979 (A) и 2978 (B), исключены 35 пользователей с неполным окном.

In [23]:
d14_events = df_eg[df_eg['event_type'] == 'return_visit_d14']
d14_num = d14_events.groupby('group')['user_id'].nunique()
denom = {'A': 2979, 'B': 2978}
d14_num / d14_num.index.map(denom)

group
A    0.289359
B    0.364338
dtype: float64

## Статистические тесты

z-тест для двух пропорций, pooled variance. Формула:

z = (p_B − p_A) / sqrt(p_pool · (1 − p_pool) · (1/n_A + 1/n_B))

p-value двусторонний через `stats.norm.sf`.

In [24]:
# Completion Rate
n_A = 2997
n_B = 2995
succ_A = 2926
succ_B = 2960
p_A = succ_A / n_A
p_B = succ_B / n_B
p_pool = (succ_A + succ_B) / (n_A + n_B)
SE = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))
z = (p_B - p_A) / SE
p_value = 2 * stats.norm.sf(abs(z))
print(f'Completion Rate: delta={p_B - p_A:.4f}, z={z:.3f}, p={p_value:.2e}')

Completion Rate: delta=0.0120, z=3.525, p=4.24e-04


In [25]:
# Retention 7
n_A = 2997
n_B = 2995
succ_A = 2179
succ_B = 2335
p_A = succ_A / n_A
p_B = succ_B / n_B
p_pool = (succ_A + succ_B) / (n_A + n_B)
SE = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))
z = (p_B - p_A) / SE
p_value = 2 * stats.norm.sf(abs(z))
print(f'Retention 7: delta={p_B - p_A:.4f}, z={z:.3f}, p={p_value:.2e}')

Retention 7: delta=0.0526, z=4.720, p=2.36e-06


In [26]:
# Retention 14
n_A = 2979
n_B = 2978
succ_A = 862
succ_B = 1085
p_A = succ_A / n_A
p_B = succ_B / n_B
p_pool = (succ_A + succ_B) / (n_A + n_B)
SE = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))
z = (p_B - p_A) / SE
p_value = 2 * stats.norm.sf(abs(z))
print(f'Retention 14: delta={p_B - p_A:.4f}, z={z:.3f}, p={p_value:.2e}')

Retention 14: delta=0.0750, z=6.169, p=6.88e-10


## Доверительные интервалы

95% CI для разности долей (p_B − p_A), unpooled SE. Формула: delta ± z_crit · SE_diff, где z_crit = stats.norm.ppf(0.975).

In [27]:
# Completion Rate
n_A = 2997
n_B = 2995
succ_A = 2926
succ_B = 2960
p_A = succ_A / n_A
p_B = succ_B / n_B
delta = p_B - p_A
SE_diff = np.sqrt(p_A*(1-p_A)/n_A + p_B*(1-p_B)/n_B)
z_crit = stats.norm.ppf(0.975)
ci_low = delta - z_crit * SE_diff
ci_high = delta + z_crit * SE_diff
print(f"Completion Rate: delta={delta:.4f}, CI ≈ [{ci_low:+.4f}; {ci_high:+.4f}]")

Completion Rate: delta=0.0120, CI ≈ [+0.0053; +0.0187]


In [28]:
# Retention 7
n_A = 2997
n_B = 2995
succ_A = 2179
succ_B = 2335
p_A = succ_A / n_A
p_B = succ_B / n_B
delta = p_B - p_A
SE_diff = np.sqrt(p_A*(1-p_A)/n_A + p_B*(1-p_B)/n_B)
z_crit = stats.norm.ppf(0.975)
ci_low = delta - z_crit * SE_diff
ci_high = delta + z_crit * SE_diff
print(f"Retention 7: delta={delta:.4f}, CI ≈ [{ci_low:+.4f}; {ci_high:+.4f}]")

Retention 7: delta=0.0526, CI ≈ [+0.0308; +0.0744]


In [29]:
# Retention 14
n_A = 2979
n_B = 2978
succ_A = 862
succ_B = 1085
p_A = succ_A / n_A
p_B = succ_B / n_B
delta = p_B - p_A
SE_diff = np.sqrt(p_A*(1-p_A)/n_A + p_B*(1-p_B)/n_B)
z_crit = stats.norm.ppf(0.975)
ci_low = delta - z_crit * SE_diff
ci_high = delta + z_crit * SE_diff
print(f"Retention 14: delta={delta:.4f}, CI ≈ [{ci_low:+.4f}; {ci_high:+.4f}]")

Retention 14: delta=0.0750, CI ≈ [+0.0512; +0.0987]


## MDE

MDE это минимальный эффект, который тест способен обнаружить при данной выборке, α = 0.05 и мощности 80%.

In [30]:
# Completion Rate
n_A = 2997
n_B = 2995
succ_A = 2926
succ_B = 2960
p_A = succ_A / n_A
p_B = succ_B / n_B
n = (n_A + n_B) / 2
z_alpha = stats.norm.ppf(1 - 0.05/2)
z_beta = stats.norm.ppf(0.80)
MDE = (z_alpha + z_beta) * np.sqrt(2 * p_A * (1 - p_A) / n)
print(f"Completion Rate: MDE ≈ {MDE:.4f}")

Completion Rate: MDE ≈ 0.0110


In [31]:
# Retention 7
n_A = 2997
n_B = 2995
succ_A = 2179
succ_B = 2335
p_A = succ_A / n_A
p_B = succ_B / n_B
n = (n_A + n_B) / 2
z_alpha = stats.norm.ppf(1 - 0.05/2)
z_beta = stats.norm.ppf(0.80)
MDE = (z_alpha + z_beta) * np.sqrt(2 * p_A * (1 - p_A) / n)
print(f"Retention 7: MDE ≈ {MDE:.4f}")

Retention 7: MDE ≈ 0.0322


In [32]:
# Retention 14
n_A = 2979
n_B = 2978
succ_A = 862
succ_B = 1085
p_A = succ_A / n_A
p_B = succ_B / n_B
n = (n_A + n_B) / 2
z_alpha = stats.norm.ppf(1 - 0.05/2)
z_beta = stats.norm.ppf(0.80)
MDE = (z_alpha + z_beta) * np.sqrt(2 * p_A * (1 - p_A) / n)
print(f"Retention 14: MDE ≈ {MDE:.4f}")

Retention 14: MDE ≈ 0.0329


MDE показывает, что тест обладал достаточной мощностью для обнаружения эффекта на Retention, но на Completion Rate эффект находится на грани: метрика ненадёжна для выводов в этой зоне.
Тест с такой выборкой надёжно фиксирует эффект на метриках Retention, а также ловит Completion Rate, но с минимальным запасом (на 0.1 п.п. выше порога). Это объясняет, почему визуально у Completion Rate разница слабая и почему метрика у потолка плохо различается.

## Проверка устойчивости

Retention 7 по бакетам стажа. Цель: убедиться, что эффект не объясняется перекосом состава групп по стажу (p = 0.015).

In [33]:
df_ev2 = df_ug[['user_id','tenure_bucket']].merge(df_eg, on='user_id', how='left')
d7 = df_ev2[df_ev2['event_type'] == 'return_visit_d7']

num_series = d7.groupby(['tenure_bucket','group'])['user_id'].nunique()
denom_series = pd.crosstab(df_ug['tenure_bucket'], df_ug['group']).stack()
rates = num_series / denom_series
rates.unstack()

group,A,B
tenure_bucket,,
новички,0.714912,0.774621
продвинутые,0.734560,0.766571
опытные,0.726966,0.786238


## Анализ воронки

Для воронки считаем уникальных пользователей, а не события: если один человек выполнил действие много раз, а другой - ни разу, агрегация по событиям исказит картину. Нам важно понять, какая доля людей от всей группы проходит каждый шаг, и на каком этапе происходит отсев. 
Дополнительно считаем конверсию шаг-к-шагу (Login→Started, Started→Completed), чтобы увидеть, где именно теряются пользователи. 
Знаменатель это все пользователи группы, чтобы увидеть, какой процент был привлечён новой фичей и как менялась доля проходящих воронку. 
Если пользователь залогинился, но не начал домашнее задание, он остаётся в знаменателе группы, но выпадает из числителя следующего шага.

Результат: таблица с уникальными юзерами по каждому шагу и конверсия шаг-к-шагу по обеим группам.

In [34]:
steps = ['login', 'homework_started', 'homework_completed']
funnel = df_eg[df_eg['event_type'].isin(steps)].groupby(['event_type', 'group'])['user_id'].nunique().unstack()
funnel = funnel.reindex(['login', 'homework_started', 'homework_completed'])
funnel.loc['Login-Started, %'] = (funnel.loc['homework_started'] / funnel.loc['login']) * 100  
funnel.loc['Started-Completed, %'] = round((funnel.loc['homework_completed'] / funnel.loc['homework_started']) * 100, 2)
funnel

group,A,B
event_type,,
login,2997.00,2995.00
homework_started,2997.00,2995.00
homework_completed,2926.00,2960.00
"Login-Started, %",100.00,100.00
"Started-Completed, %",97.63,98.83


По результатам анализа воронки можно сказать, что все пользователи в группах прошли первый и второй шаг, разница видна лишь на третьем, а именно завершении начатой домашней работы. 
У группы В показатели выше на 1.2 п.п., что совпадает с Completion Rate из основной таблицы.
Воронка помогает подтвердить, что фича оказывает умеренное влияние на завершение уже начатой работы. 
Поскольку разница между группами возникает только на этапе завершения, можно предположить, что геймификация не удерживает пользователей на входе в продукт, а стимулирует их возвращаться к незавершённым задачам, что согласуется с ростом Retention 7 и 14.

## Сегментация по классам

Post-hoc анализ: проверяем, одинаково ли работает фича в младших, средних и старших классах. Exploratory-анализ, p-values не скорректированы на множественные сравнения.

In [35]:
def grade_bucket(g):
    if g <= 7:
        return '5-7'
    elif g <= 9:
        return '8-9'
    else:
        return '10-11'
num_g = (df_eg[df_eg['event_type'] == 'return_visit_d14']
         .groupby([df_eg['grade'].apply(grade_bucket), 'group'])['user_id']
         .nunique())
num_g

grade  group
10-11  A        239
       B        219
5-7    A        366
       B        585
8-9    A        257
       B        281
Name: user_id, dtype: int64

In [36]:
d14z['grade_group'] = d14z['grade'].apply(grade_bucket)
denom_g = d14z.groupby(['grade_group', 'group'])['user_id'].nunique()


In [37]:
num_g / denom_g

grade  group
10-11  A        0.293612
       B        0.256440
5-7    A        0.284161
       B        0.444867
8-9    A        0.293044
       B        0.347342
Name: user_id, dtype: float64

In [38]:
# 10-11 grade
n_A = 814
n_B = 854
succ_A = 239 
succ_B = 219
p_A = succ_A / n_A
p_B = succ_B / n_B
p_pool = (succ_A + succ_B) / (n_A + n_B)
SE = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))
z = (p_B - p_A) / SE
p_value = 2 * stats.norm.sf(abs(z))
print(f'10-11 grade: delta={p_B - p_A:.4f}, z={z:.3f}, p={p_value:.2e}')

10-11 grade: delta=-0.0372, z=-1.700, p=8.91e-02


In [39]:
# 5-7 grade
n_A=1288
n_B=1315 
succ_A=366 
succ_B=585
p_A = succ_A / n_A
p_B = succ_B / n_B
p_pool = (succ_A + succ_B) / (n_A + n_B)
SE = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))
z = (p_B - p_A) / SE
p_value = 2 * stats.norm.sf(abs(z))
print(f'5-7 grade: delta={p_B - p_A:.4f}, z={z:.3f}, p={p_value:.2e}')

5-7 grade: delta=0.1607, z=8.513, p=1.69e-17


In [40]:
# 8-9 grade
n_A=877
n_B=809 
succ_A=257 
succ_B=281
p_A = succ_A / n_A
p_B = succ_B / n_B
p_pool = (succ_A + succ_B) / (n_A + n_B)
SE = np.sqrt(p_pool * (1 - p_pool) * (1/n_A + 1/n_B))
z = (p_B - p_A) / SE
p_value = 2 * stats.norm.sf(abs(z))
print(f'8-9 grade: delta={p_B - p_A:.4f}, z={z:.3f}, p={p_value:.2e}')

8-9 grade: delta=0.0543, z=2.390, p=1.69e-02


Разбивка по классам проведена после основного анализа, это exploratory-проверка. P-values в трёх подгруппах не скорректированы на множественные сравнения, поэтому результат по 10-11 стоит интерпретировать с осторожностью: 5-7 +16.1 п.п. (p < 0.001), 8-9 +5.4 п.п. (p = 0.017), 10-11 −3.7 п.п. (p = 0.089).
Общий эффект на Retention 14 (+7.5 п.п., p = 6.9e-10) разбивка не отменяет, а дополняет: в младших и средних классах фича работает, в старших данных недостаточно, чтобы утверждать вред.
Рекомендация: раскатывать полностью, старшие классы требуют отдельного мониторинга.

## Вывод

Фича показывает устойчивый положительный эффект на всех трёх метриках: Completion Rate +1.20 п.п. (p = 4.2e-04), Retention 7 +5.26 п.п. (p = 2.4e-06), Retention 14 +7.50 п.п. (p = 6.9e-10). Все три результата статистически значимы, 95% доверительные интервалы не накрывают ноль. Эффект растёт с удлинением горизонта, что указывает на влияние геймификации прежде всего на возвращаемость, а не на разовое действие.

**Устойчивость.** Эффект на Retention 7 сохраняется во всех трёх бакетах стажа: +5.97 п.п. у новичков, +3.20 п.п. у продвинутых, +5.93 п.п. у опытных. Перекос по стажу (p = 0.015) не объясняет разницу. Разбивка по классам показывает неоднородность: +16.1 п.п. в 5-7 (p < 0.001), +5.4 п.п. в 8-9 (p = 0.017), −3.7 п.п. в 10-11 (p = 0.089). В младших классах эффект сильный, в старших данных недостаточно для вывода.

**Ограничения.** MDE для Completion Rate (1.1 п.п.) почти совпадает с фактическим эффектом (1.2 п.п.), метрика у потолка и ненадёжна для выводов. Горизонт наблюдения 20 дней, долгосрочный эффект не проверялся. Смежные метрики не измерялись.

**Рекомендация.** Раскатить фичу на 100% пользователей. После раскатки мониторинг Retention 30/60 у новых когорт раз в 1–2 месяца, сравнение с базовой линией периода эксперимента. Старшие классы (10-11) требуют отдельного мониторинга.